# XGBoost Model Development and Optimization

**Module:** IT3051 – Fundamentals of Data Mining  
**Project:** Hotel Booking Cancellation Prediction  
**Member:** Member 04  
**Algorithm:** XGBoost Classifier  

## Objective

The objective of this notebook is to develop, evaluate, and optimize an XGBoost classification model for predicting hotel booking cancellations.

The XGBoost model will first be evaluated using a baseline configuration with 5-fold Stratified Cross-Validation. Hyperparameter tuning will then be performed using only the training data. The baseline and tuned models will be compared before selecting and freezing the final configuration for evaluation on the independent hold-out test set.

In [1]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [2]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from xgboost import XGBClassifier

In [3]:
df = pd.read_csv(
    "../data/processed/feature_engineered_dataset.csv"
)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (119390, 35)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_nights,total_guests,is_family,service_score,customer_history,room_changed
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,0,2.0,0,0,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,0,2.0,0,0,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,1,1.0,0,0,0,1
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,1,1.0,0,0,0,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,2,2.0,0,1,0,0


In [4]:
print("Target column exists:", "is_canceled" in df.columns)

print("\nTarget distribution:")
print(df["is_canceled"].value_counts())

print("\nTarget percentage distribution:")
print(
    df["is_canceled"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Target column exists: True

Target distribution:
is_canceled
0    75166
1    44224
Name: count, dtype: int64

Target percentage distribution:
is_canceled
0    62.96
1    37.04
Name: proportion, dtype: float64


In [5]:
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

Predictor shape: (119390, 34)
Target shape: (119390,)


In [6]:
categorical_cols = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Number of categorical features:", len(categorical_cols))
print("Number of numerical features:", len(numerical_cols))

print("\nCategorical features:")
print(categorical_cols)

print("\nNumerical features:")
print(numerical_cols)

Number of categorical features: 10
Number of numerical features: 24

Categorical features:
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']

Numerical features:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests', 'is_family', 'service_score', 'customer_history', 'room_changed']


## 2. Train-Test Split and Cross-Validation Setup

The dataset is divided into 80% training data and 20% final hold-out test data using stratified sampling.

A fixed `random_state=42` is used for reproducibility, while stratification preserves the proportion of cancelled and non-cancelled bookings in both subsets.

The final test set is kept completely separate from model development and hyperparameter tuning.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTraining target distribution (%):")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution (%):")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training records: 95512
Testing records: 23878

Training target distribution (%):
is_canceled
0    62.96
1    37.04
Name: proportion, dtype: float64

Testing target distribution (%):
is_canceled
0    62.96
1    37.04
Name: proportion, dtype: float64


## 3. XGBoost Preprocessing Pipeline

Categorical predictors are transformed using One-Hot Encoding with `handle_unknown="ignore"` so that they can be processed by the XGBoost classifier.

Numerical predictors are passed through without scaling because XGBoost is based on decision-tree boosting and does not require numerical features to be standardized.

The preprocessing steps and XGBoost classifier are combined in a Pipeline so that preprocessing is fitted correctly within each cross-validation fold and the risk of data leakage is reduced.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "numerical",
            "passthrough",
            numerical_cols
        )
    ]
)

print("XGBoost preprocessor created successfully.")

XGBoost preprocessor created successfully.


In [10]:
baseline_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [11]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", baseline_xgb)
    ]
)

print("Baseline XGBoost pipeline created successfully.")

Baseline XGBoost pipeline created successfully.


## 4. Baseline XGBoost Evaluation

The baseline XGBoost classifier is evaluated before hyperparameter tuning to establish a reference performance level.

Five-fold Stratified Cross-Validation is performed using only the training data. Accuracy, Precision, Recall, F1-score, and ROC-AUC are evaluated.

Training scores are also recorded and compared with cross-validation scores to investigate model generalization and possible overfitting.

In [12]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

print("Evaluation metrics:", list(scoring.keys()))

Evaluation metrics: ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']


In [13]:
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [14]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Cross-validation strategy:")
print(cv)

Cross-validation strategy:
StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [15]:
baseline_results = {
    "train_accuracy": [], "test_accuracy": [],
    "train_precision": [], "test_precision": [],
    "train_recall": [], "test_recall": [],
    "train_f1": [], "test_f1": [],
    "train_roc_auc": [], "test_roc_auc": []
}

baseline_start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train),
    start=1
):
    print(f"\nRunning fold {fold}/5...")

    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    model = clone(baseline_model)

    model.fit(X_fold_train, y_fold_train)

    train_pred = model.predict(X_fold_train)
    val_pred = model.predict(X_fold_val)

    train_prob = model.predict_proba(X_fold_train)[:, 1]
    val_prob = model.predict_proba(X_fold_val)[:, 1]

    baseline_results["train_accuracy"].append(
        accuracy_score(y_fold_train, train_pred)
    )
    baseline_results["test_accuracy"].append(
        accuracy_score(y_fold_val, val_pred)
    )

    baseline_results["train_precision"].append(
        precision_score(y_fold_train, train_pred, zero_division=0)
    )
    baseline_results["test_precision"].append(
        precision_score(y_fold_val, val_pred, zero_division=0)
    )

    baseline_results["train_recall"].append(
        recall_score(y_fold_train, train_pred, zero_division=0)
    )
    baseline_results["test_recall"].append(
        recall_score(y_fold_val, val_pred, zero_division=0)
    )

    baseline_results["train_f1"].append(
        f1_score(y_fold_train, train_pred, zero_division=0)
    )
    baseline_results["test_f1"].append(
        f1_score(y_fold_val, val_pred, zero_division=0)
    )

    baseline_results["train_roc_auc"].append(
        roc_auc_score(y_fold_train, train_prob)
    )
    baseline_results["test_roc_auc"].append(
        roc_auc_score(y_fold_val, val_prob)
    )

    print(f"Fold {fold}/5 completed.")

baseline_cv_time = time.time() - baseline_start_time

print("\nAll 5 baseline folds completed!")
print(f"Baseline CV time: {baseline_cv_time:.2f} seconds")


Running fold 1/5...
Fold 1/5 completed.

Running fold 2/5...
Fold 2/5 completed.

Running fold 3/5...
Fold 3/5 completed.

Running fold 4/5...
Fold 4/5 completed.

Running fold 5/5...
Fold 5/5 completed.

All 5 baseline folds completed!
Baseline CV time: 11.84 seconds


In [16]:
import pandas as pd
import numpy as np

In [17]:
# Baseline XGBoost CV results summary

baseline_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],

    "Train Mean": [
        np.mean(baseline_results["train_accuracy"]),
        np.mean(baseline_results["train_precision"]),
        np.mean(baseline_results["train_recall"]),
        np.mean(baseline_results["train_f1"]),
        np.mean(baseline_results["train_roc_auc"])
    ],

    "Train Std": [
        np.std(baseline_results["train_accuracy"]),
        np.std(baseline_results["train_precision"]),
        np.std(baseline_results["train_recall"]),
        np.std(baseline_results["train_f1"]),
        np.std(baseline_results["train_roc_auc"])
    ],

    "Validation Mean": [
        np.mean(baseline_results["test_accuracy"]),
        np.mean(baseline_results["test_precision"]),
        np.mean(baseline_results["test_recall"]),
        np.mean(baseline_results["test_f1"]),
        np.mean(baseline_results["test_roc_auc"])
    ],

    "Validation Std": [
        np.std(baseline_results["test_accuracy"]),
        np.std(baseline_results["test_precision"]),
        np.std(baseline_results["test_recall"]),
        np.std(baseline_results["test_f1"]),
        np.std(baseline_results["test_roc_auc"])
    ]
})

baseline_summary["Train-Val Gap"] = (
    baseline_summary["Train Mean"] -
    baseline_summary["Validation Mean"]
)

print("Baseline XGBoost - 5-Fold CV Summary")
print(f"CV Time: {baseline_cv_time:.2f} seconds")

display(baseline_summary.round(4))

Baseline XGBoost - 5-Fold CV Summary
CV Time: 11.84 seconds


,Metric,Train Mean,Train Std,Validation Mean,Validation Std,Train-Val Gap
0,Accuracy,0.8976,0.0012,0.8813,0.0020,0.0164
1,Precision,0.8788,0.0013,0.8569,0.0020,0.0219
2,Recall,0.8395,0.0025,0.8158,0.0056,0.0237
3,F1 Score,0.8587,0.0018,0.8358,0.0032,0.0229
4,ROC-AUC,0.9657,0.0008,0.9528,0.0011,0.0129


## 5. XGBoost Hyperparameter Tuning - Full Grid Search

In [18]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200, 300, 400],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.8, 0.9, 1.0]
}

print("Full XGBoost parameter grid configured.")

Full XGBoost parameter grid configured.


In [19]:
%pip install tqdm tqdm-joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=baseline_model,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=0,
    return_train_score=True
)

print("GridSearchCV configured successfully.")
print("Parameter combinations: 2160")
print("CV folds: 5")
print("Total model fits: 10800")

GridSearchCV configured successfully.
Parameter combinations: 2160
CV folds: 5
Total model fits: 10800


In [21]:
import time
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib

total_fits = 2160 * 5

tuning_start_time = time.time()

with tqdm_joblib(
    tqdm(
        total=total_fits,
        desc="XGBoost Grid Search",
        unit="fit"
    )
):
    grid_search.fit(X_train, y_train)

tuning_time = time.time() - tuning_start_time

print("\nHyperparameter tuning completed!")
print(f"Total tuning time: {tuning_time / 60:.2f} minutes")

XGBoost Grid Search:   0%|          | 0/10800 [00:00<?, ?fit/s]

  0%|          | 0/10800 [00:00<?, ?it/s]


Hyperparameter tuning completed!
Total tuning time: 120.56 minutes


In [22]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(f"{grid_search.best_score_:.4f}")

best_xgb_model = grid_search.best_estimator_

print("\nBest XGBoost model successfully extracted.")

Best Parameters:
{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': 8, 'model__min_child_weight': 1, 'model__n_estimators': 400, 'model__subsample': 0.8}

Best Cross-Validation F1 Score:
0.8498

Best XGBoost model successfully extracted.


In [23]:
# Final evaluation of the tuned XGBoost model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Predictions using the best tuned model
y_pred_tuned = best_xgb_model.predict(X_test)
y_prob_tuned = best_xgb_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_precision = precision_score(y_test, y_pred_tuned)
tuned_recall = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)
tuned_roc_auc = roc_auc_score(y_test, y_prob_tuned)

# Display results
print("Tuned XGBoost Test Set Performance")
print("=" * 40)
print(f"Accuracy  : {tuned_accuracy:.4f}")
print(f"Precision : {tuned_precision:.4f}")
print(f"Recall    : {tuned_recall:.4f}")
print(f"F1 Score  : {tuned_f1:.4f}")
print(f"ROC-AUC   : {tuned_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

Tuned XGBoost Test Set Performance
Accuracy  : 0.8925
Precision : 0.8688
Recall    : 0.8358
F1 Score  : 0.8520
ROC-AUC   : 0.9611

Confusion Matrix:
[[13917  1116]
 [ 1452  7393]]


In [25]:
import joblib
import os

# Create folder for saved models
os.makedirs("models", exist_ok=True)

# Extract the best tuned model
best_xgb_model = grid_search.best_estimator_

# Save the complete trained pipeline/model
model_path = "models/xgboost_tuned_model.pkl"
joblib.dump(best_xgb_model, model_path)

print("Tuned XGBoost model saved successfully!")
print("Saved to:", model_path)

Tuned XGBoost model saved successfully!
Saved to: models/xgboost_tuned_model.pkl


In [29]:
# ==========================================
# BASELINE VS TUNED XGBOOST COMPARISON TABLE
# ==========================================

# Baseline XGBoost 5-fold CV mean scores
baseline_cv_accuracy = 0.8813
baseline_cv_precision = 0.8569
baseline_cv_recall = 0.8158
baseline_cv_f1 = 0.8358
baseline_cv_roc_auc = 0.9528

comparison_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],

    "Baseline XGBoost": [
        baseline_cv_accuracy,
        baseline_cv_precision,
        baseline_cv_recall,
        baseline_cv_f1,
        baseline_cv_roc_auc
    ],

    "Tuned XGBoost": [
        tuned_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1,
        tuned_roc_auc
    ]
})

# Calculate improvement
comparison_df["Improvement"] = (
    comparison_df["Tuned XGBoost"] -
    comparison_df["Baseline XGBoost"]
)

# Round values for readability
comparison_df = comparison_df.round(4)

print("Baseline vs Tuned XGBoost Performance Comparison")
display(comparison_df)

Baseline vs Tuned XGBoost Performance Comparison


,Metric,Baseline XGBoost,Tuned XGBoost,Improvement
0,Accuracy,0.8813,0.8925,0.0112
1,Precision,0.8569,0.8688,0.0119
2,Recall,0.8158,0.8358,0.0200
3,F1 Score,0.8358,0.8520,0.0162
4,ROC-AUC,0.9528,0.9611,0.0083


In [30]:
# Save comparison table as CSV
comparison_df.to_csv("xgboost_comparison_table.csv", index=False)

print("Comparison table saved as xgboost_comparison_table.csv")

Comparison table saved as xgboost_comparison_table.csv
